# 15 — Evaluation and bootstrap sensitivity

This post-protocol analysis uses retained development and external-panel forecasts to replace pointwise-only inference with simultaneous family-wise intervals and to propagate predictive-draw Monte Carlo error. The seed/date bootstrap and draw-axis bootstrap are combined as independent variance components. Matched neural models use paired draw indices; unrelated neural and VAR–GARCH forecasts use independent draw indices. It also checks draw-count and block-length sensitivity. Exact test NLL and analytic Gaussian projected PITs still require retained distribution parameters or refitting.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from innovcal.evaluation import (
    combined_seed_draw_bootstrap, evaluate_samples, paired_draw_mc_sensitivity,
)

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'results' / 'evaluation_sensitivity'
OUT.mkdir(parents=True, exist_ok=True)
ARCHIVE = np.load(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecasts.npz')
MANIFEST = pd.read_csv(ROOT / 'results' / 'seed_uncertainty' / 'seed_forecast_manifest.csv')
full_label = 'CA-RNN sequential' if 'CA-RNN sequential' in set(MANIFEST.model) else 'CA-RNN'
keys = dict(zip(MANIFEST.model, MANIFEST.array_key))
target = ARCHIVE['target']
projections = ARCHIVE['projections']
forecasts = {'RNN': ARCHIVE[keys['RNN']], 'CA-RNN': ARCHIVE[keys[full_label]]}
assert forecasts['RNN'].shape == forecasts['CA-RNN'].shape
print('Matched seeds, draws, origins, dimensions:', forecasts['RNN'].shape)

Matched seeds, draws, origins, dimensions: (10, 500, 957, 4)


In [2]:
rows = []
for model, all_seeds in forecasts.items():
    for seed_index, samples in enumerate(all_seeds):
        values = evaluate_samples(target, samples, projections=projections)
        assert np.isclose(values['rmse'] ** 2, values['mean_squared_error'])
        rows.append({'model': model, 'seed_index': seed_index, **values})
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT / 'seed_metric_audit.csv', index=False)
display(metrics.groupby('model')[['mean_squared_error', 'rmse', 'energy_score', 'variogram_score']].mean())
print('MSE is mean squared component error; RMSE is its square root computed within each seed.')

,mean_squared_error,rmse,energy_score,variogram_score
model,,,,
CA-RNN,0.898605,0.947946,1.153081,0.156705
RNN,0.883330,0.939856,1.145041,0.156821


MSE is mean squared component error; RMSE is its square root computed within each seed.


In [3]:
saved_metrics = pd.read_csv(ROOT / 'results' / 'seed_uncertainty' / 'seed_metrics.csv')
new_columns = ['test_nll', 'analytic_pit_calibration_error', 'analytic_pit_mean_absolute_autocorrelation']
if set(new_columns).issubset(saved_metrics.columns):
    full_model = 'CA-RNN sequential' if 'CA-RNN sequential' in set(saved_metrics.model) else 'CA-RNN'
    analytic = saved_metrics.loc[saved_metrics.model.isin(['RNN', full_model]), ['model', 'training_seed', *new_columns]].copy()
    analytic.loc[analytic.model.eq(full_model), 'model'] = 'CA-RNN'
    analytic.to_csv(OUT / 'analytic_pit_and_test_nll.csv', index=False)
    display(analytic.groupby('model')[new_columns].mean())
else:
    print('Saved fits predate analytic PIT and test NLL fields. Rerun notebook 08 to populate them; sample draws cannot recover exact Gaussian parameters.')

,test_nll,analytic_pit_calibration_error,analytic_pit_mean_absolute_autocorrelation
model,,,
CA-RNN,5.309488,0.001223,0.036909
RNN,5.271313,0.001439,0.030724


In [5]:
PRIMARY_METRICS = ('mean_squared_error', 'energy_score', 'variogram_score', 'interval_score', 'coverage_error', 'pit_calibration_error', 'pit_mean_absolute_autocorrelation')
development_inference = combined_seed_draw_bootstrap(
    target, forecasts, [('CA-RNN', 'RNN')], projections,
    block_length=20, n_bootstrap=2000, n_draw_resamples=200,
    confidence=0.95, seed=808, draw_seed=1608, draw_pairing='paired',
    simultaneous_metrics=PRIMARY_METRICS, familywise_confidence=0.95,
)
development_inference.to_csv(OUT / 'development_combined_inference.csv', index=False)
development_draw_mc = paired_draw_mc_sensitivity(
    target, forecasts, [('CA-RNN', 'RNN')], projections,
    draw_counts=(100, 250, 500), n_resamples=100, seed=1615, metrics=PRIMARY_METRICS,
)
development_draw_mc.to_csv(OUT / 'development_draw_mc_sensitivity.csv', index=False)

frames = []
for block_length in (10, 20, 40):
    result = combined_seed_draw_bootstrap(
        target, forecasts, [('CA-RNN', 'RNN')], projections,
        block_length=block_length, n_bootstrap=1000, n_draw_resamples=100,
        confidence=0.95, seed=1500 + block_length, draw_seed=2500 + block_length,
        draw_pairing='paired', simultaneous_metrics=PRIMARY_METRICS, familywise_confidence=0.95,
    )
    frames.append(result)
sensitivity = pd.concat(frames, ignore_index=True)
sensitivity['excludes_zero'] = sensitivity.combined_familywise_resolved.fillna(False)
sensitivity.to_csv(OUT / 'block_length_sensitivity.csv', index=False)

confirmatory_archive = np.load(ROOT / 'results' / 'confirmatory' / 'confirmatory_forecasts.npz')
confirmatory_manifest = pd.read_csv(ROOT / 'results' / 'confirmatory' / 'forecast_manifest.csv')
confirmatory_keys = dict(zip(confirmatory_manifest.model, confirmatory_manifest.array_key))
confirmatory_label = 'CA-RNN sequential' if 'CA-RNN sequential' in confirmatory_keys else 'CA-RNN'
confirmatory_target = confirmatory_archive['target']
confirmatory_projections = confirmatory_archive['projections']
confirmatory_forecasts = {
    'RNN': confirmatory_archive[confirmatory_keys['RNN']],
    'CA-RNN': confirmatory_archive[confirmatory_keys[confirmatory_label]],
    'VAR-GARCH bootstrap': confirmatory_archive[confirmatory_keys['VAR-GARCH bootstrap']],
}
confirmatory_comparisons = [('CA-RNN', 'RNN'), ('RNN', 'VAR-GARCH bootstrap'), ('CA-RNN', 'VAR-GARCH bootstrap')]
pairing = {comparison: ('paired' if comparison == ('CA-RNN', 'RNN') else 'independent') for comparison in confirmatory_comparisons}
confirmatory_inference = combined_seed_draw_bootstrap(
    confirmatory_target, confirmatory_forecasts, confirmatory_comparisons, confirmatory_projections,
    block_length=20, n_bootstrap=2000, n_draw_resamples=200,
    confidence=0.95, seed=1208, draw_seed=1607, draw_pairing=pairing,
    simultaneous_metrics=PRIMARY_METRICS, familywise_confidence=0.95,
)
confirmatory_inference['analysis_status'] = np.where((confirmatory_inference.candidate == 'CA-RNN') & (confirmatory_inference.comparator == 'RNN'), 'primary', 'secondary')
confirmatory_inference.to_csv(OUT / 'confirmatory_combined_inference.csv', index=False)
confirmatory_draw_mc = paired_draw_mc_sensitivity(
    confirmatory_target, {'RNN': confirmatory_forecasts['RNN'], 'CA-RNN': confirmatory_forecasts['CA-RNN']},
    [('CA-RNN', 'RNN')], confirmatory_projections, draw_counts=(100, 250, 500),
    n_resamples=100, seed=1616, metrics=PRIMARY_METRICS,
)
confirmatory_draw_mc.to_csv(OUT / 'confirmatory_draw_mc_sensitivity.csv', index=False)

display(development_inference.loc[development_inference.metric.isin(PRIMARY_METRICS)])
display(confirmatory_inference.loc[confirmatory_inference.metric.isin(PRIMARY_METRICS)])
display(sensitivity.loc[sensitivity.metric.isin(PRIMARY_METRICS)])

,n_seeds,n_origins,block_length,n_bootstrap,candidate,comparator,metric,difference,bootstrap_se,ci_lower,...,draw_mc_variance_fraction,n_draw_resamples,draw_resampling,combined_interval_method,combined_familywise_ci_lower,combined_familywise_ci_upper,combined_familywise_critical_value,combined_familywise_family_size,combined_familywise_confidence,combined_familywise_resolved
0,10,957,20,2000,CA-RNN,RNN,mean_squared_error,0.015275,0.003794,0.008615,...,0.006844,200.0,paired,independence_convolution,0.003838,0.026713,3.00359,7.0,0.95,True
1,10,957,20,2000,CA-RNN,RNN,energy_score,0.008040,0.001887,0.004383,...,0.008015,200.0,paired,independence_convolution,0.002358,0.013721,3.00359,7.0,0.95,True
2,10,957,20,2000,CA-RNN,RNN,variogram_score,-0.000116,0.000436,-0.000826,...,0.010039,200.0,paired,independence_convolution,-0.001436,0.001204,3.00359,7.0,0.95,False
3,10,957,20,2000,CA-RNN,RNN,interval_score,-0.003866,0.016218,-0.037189,...,0.028844,200.0,paired,independence_convolution,-0.053473,0.045741,3.00359,7.0,0.95,False
6,10,957,20,2000,CA-RNN,RNN,coverage_error,-0.005277,0.003105,-0.010999,...,0.063498,200.0,paired,independence_convolution,-0.014798,0.004244,3.00359,7.0,0.95,False
7,10,957,20,2000,CA-RNN,RNN,pit_calibration_error,-0.000213,0.000102,-0.000468,...,0.006696,200.0,paired,independence_convolution,-0.000519,0.000093,3.00359,7.0,0.95,False
8,10,957,20,2000,CA-RNN,RNN,pit_mean_absolute_autocorrelation,0.006551,0.003205,-0.003157,...,0.008553,200.0,paired,independence_convolution,-0.003124,0.016226,3.00359,7.0,0.95,False


,n_seeds,n_origins,block_length,n_bootstrap,candidate,comparator,metric,difference,bootstrap_se,ci_lower,...,n_draw_resamples,draw_resampling,combined_interval_method,combined_familywise_ci_lower,combined_familywise_ci_upper,combined_familywise_critical_value,combined_familywise_family_size,combined_familywise_confidence,combined_familywise_resolved,analysis_status
0,10,989,20,2000,CA-RNN,RNN,mean_squared_error,0.014762,0.003539,0.008351,...,200.0,paired,independence_convolution,0.004717,0.024808,2.826231,7.0,0.95,True,primary
1,10,989,20,2000,CA-RNN,RNN,energy_score,0.010075,0.002067,0.005989,...,200.0,paired,independence_convolution,0.004206,0.015944,2.826231,7.0,0.95,True,primary
2,10,989,20,2000,CA-RNN,RNN,variogram_score,0.000270,0.000483,-0.000732,...,200.0,paired,independence_convolution,-0.001096,0.001635,2.826231,7.0,0.95,False,primary
3,10,989,20,2000,CA-RNN,RNN,interval_score,0.023177,0.016474,-0.010485,...,200.0,paired,independence_convolution,-0.023841,0.070195,2.826231,7.0,0.95,False,primary
6,10,989,20,2000,CA-RNN,RNN,coverage_error,0.002022,0.003331,-0.004198,...,200.0,paired,independence_convolution,-0.007940,0.011984,2.826231,7.0,0.95,False,primary
7,10,989,20,2000,CA-RNN,RNN,pit_calibration_error,-0.000047,0.000097,-0.000295,...,200.0,paired,independence_convolution,-0.000324,0.000229,2.826231,7.0,0.95,False,primary
8,10,989,20,2000,CA-RNN,RNN,pit_mean_absolute_autocorrelation,0.003700,0.003539,-0.003983,...,200.0,paired,independence_convolution,-0.006386,0.013785,2.826231,7.0,0.95,False,primary
9,10,989,20,2000,RNN,VAR-GARCH bootstrap,mean_squared_error,-0.006370,0.003758,-0.013872,...,200.0,independent,independence_convolution,-0.018151,0.005412,2.735408,7.0,0.95,False,secondary
10,10,989,20,2000,RNN,VAR-GARCH bootstrap,energy_score,-0.003623,0.002863,-0.008789,...,200.0,independent,independence_convolution,-0.015709,0.008463,2.735408,7.0,0.95,False,secondary
11,10,989,20,2000,RNN,VAR-GARCH bootstrap,variogram_score,-0.000511,0.001148,-0.002863,...,200.0,independent,independence_convolution,-0.003759,0.002737,2.735408,7.0,0.95,False,secondary


,n_seeds,n_origins,block_length,n_bootstrap,candidate,comparator,metric,difference,bootstrap_se,ci_lower,...,n_draw_resamples,draw_resampling,combined_interval_method,combined_familywise_ci_lower,combined_familywise_ci_upper,combined_familywise_critical_value,combined_familywise_family_size,combined_familywise_confidence,combined_familywise_resolved,excludes_zero
0,10,957,10,1000,CA-RNN,RNN,mean_squared_error,0.015275,0.003801,0.008352,...,100.0,paired,independence_convolution,0.004270,0.026281,2.890551,7.0,0.95,True,True
1,10,957,10,1000,CA-RNN,RNN,energy_score,0.008040,0.002103,0.004069,...,100.0,paired,independence_convolution,0.001943,0.014136,2.890551,7.0,0.95,True,True
2,10,957,10,1000,CA-RNN,RNN,variogram_score,-0.000116,0.000451,-0.000892,...,100.0,paired,independence_convolution,-0.001426,0.001194,2.890551,7.0,0.95,False,False
3,10,957,10,1000,CA-RNN,RNN,interval_score,-0.003866,0.017489,-0.037703,...,100.0,paired,independence_convolution,-0.054632,0.046899,2.890551,7.0,0.95,False,False
6,10,957,10,1000,CA-RNN,RNN,coverage_error,-0.005277,0.002860,-0.010659,...,100.0,paired,independence_convolution,-0.013932,0.003378,2.890551,7.0,0.95,False,False
7,10,957,10,1000,CA-RNN,RNN,pit_calibration_error,-0.000213,0.000102,-0.000466,...,100.0,paired,independence_convolution,-0.000509,0.000082,2.890551,7.0,0.95,False,False
8,10,957,10,1000,CA-RNN,RNN,pit_mean_absolute_autocorrelation,0.006551,0.003310,-0.002746,...,100.0,paired,independence_convolution,-0.003027,0.016129,2.890551,7.0,0.95,False,False
9,10,957,20,1000,CA-RNN,RNN,mean_squared_error,0.015275,0.003815,0.008728,...,100.0,paired,independence_convolution,0.003252,0.027299,3.147390,7.0,0.95,True,True
10,10,957,20,1000,CA-RNN,RNN,energy_score,0.008040,0.001909,0.004401,...,100.0,paired,independence_convolution,0.002013,0.014066,3.147390,7.0,0.95,True,True
11,10,957,20,1000,CA-RNN,RNN,variogram_score,-0.000116,0.000437,-0.000801,...,100.0,paired,independence_convolution,-0.001496,0.001264,3.147390,7.0,0.95,False,False


The revised primary inference uses the original 20-day block length and 2,000 seed/date replicates, adds 200 predictive-draw resamples, and reports 95% max-|z| simultaneous intervals over the prespecified seven-metric family within each comparison. Draw-count and block-length analyses are sensitivity checks. Analytic PIT and NLL values require retained Gaussian parameters or refitting and are not approximated from forecast samples. Financial conclusions remain specific to the observed panels; broader controlled-DGP inference comes from the crossed isolated-shift experiment, which refits models across independent datasets and neural seeds.